In [ ]:
import pandas as pd
import numpy as np

In [ ]:
ecotox_data = pd.read_csv("../Output/ecotox_test_results.tsv", sep="\t", dtype=str)
ecotox_data = ecotox_data.replace(np.nan, '', regex=True)
print(ecotox_data.shape)
ecotox_data.head()

In [ ]:
list(ecotox_data.columns)

In [ ]:
ecotox_data['source_year'] = ecotox_data['source'] + ". Year:" + ecotox_data['publication_year']
print(ecotox_data.shape)
ecotox_data.head()

In [ ]:
ecotox_data['doi'].value_counts()

In [ ]:
ecotox_data['doi'] = ecotox_data['doi'].apply(lambda x: 'DOI:'+str(x) if x!='' else x)
print(ecotox_data.shape)
ecotox_data.head()


In [ ]:
ecotox_data['ref'] = ecotox_data.apply(lambda row: row['source_year'] if row['doi']=='' else row['doi'], axis=1)
print(ecotox_data.shape)
ecotox_data.head()

In [ ]:
ecotox_data['ref'].value_counts()

In [ ]:
ecotox_data = pd.DataFrame(ecotox_data[[
'result_id',
'test_id',
'reference_number',
'test_cas',
'sample_size_mean_op',
'sample_size_mean',
'sample_size_unit',
'obs_duration_mean_op',
'obs_duration_mean',
'obs_duration_unit',
'endpoint',
'endpoint_comments',
'trend',
'trend_description',
'effect',
'effect_description',
'effect_comments',
'measurement',
'measurement_comments',
'measurement_description',
'response_site',
'response_site_comments',
'effect_pct_mean_op',
'effect_pct_mean',
'halflife_mean_op',
'halflife_mean',
'halflife_unit',
'conc1_type',
'conc1_mean_op',
'conc1_mean',
'conc1_unit',
'conc2_type',
'conc2_mean_op',
'conc2_mean',
'conc2_unit',
'conc3_type',
'conc3_mean_op',
'conc3_mean',
'conc3_unit',
'bcf1_mean_op',
'bcf1_mean',
'bcf1_unit',
'bcf2_mean_op',
'bcf2_mean',
'bcf2_unit',
'bcf3_mean_op',
'bcf3_mean',
'bcf3_unit',
'significance_code',
'significance_type',
'significance_level_mean_op',
'significance_level_mean',
'species_number',
'organism_habitat',
'organism_source',
'organism_source_comments',
'organism_lifestage',
'organism_lifestage_comments',
'study_duration_mean_op',
'study_duration_mean',
'study_duration_unit',
'exposure_duration_mean_op',
'exposure_duration_mean',
'exposure_duration_unit',
'study_type',
'exposure_type',
'test_type',
'control_type',
'num_doses_mean_op',
'num_doses_mean',
'media_type',
'media_type_comments',
'subhabitat',
'subhabitat_description',
'geographic_code',
'geographic_location',
'water_depth_mean_op',
'water_depth_mean',
'water_depth_unit',
'latitude',
'longitude',
'chemical_name',
'dtxsid',
'common_name',
'latin_name',
'ecotox_group_x',
'ecotox_group_y',
'ncbi_taxid',
'source_year',
'doi',
'ref'
]])

print(ecotox_data.shape)
ecotox_data.head()

In [ ]:
ecotox_data['measurement']

In [ ]:
ecotox_data['measurement'] = ecotox_data['measurement'].str.replace('/', '', regex=True)
print(ecotox_data.shape)
ecotox_data.head()

In [ ]:
ecotox_data.rename(columns={"ecotox_group_x":"ecotox_group_chem", "ecotox_group_y":"ecotox_group_species"}, inplace=True)
print(ecotox_data.shape)
ecotox_data.head()

In [ ]:
ecotox_data = ecotox_data.replace(np.nan, '', regex=True)
print(ecotox_data.shape)
ecotox_data.drop_duplicates(inplace=True)
print(ecotox_data.shape)
ecotox_data.head()

In [ ]:
ecotox_data['ecotox_group_species'] = ecotox_data['ecotox_group_species'].str.replace(';Standard Test Species','').str.replace(';U.S. Threatened and Endangered Species', '').str.replace(';U.S. Invasive Species', '')
print(ecotox_data.shape)
ecotox_data['endpoint'] = ecotox_data['endpoint'].str.replace('*','').str.replace('/', '')
ecotox_data.head()

In [ ]:
set(ecotox_data['ecotox_group_species'])

In [ ]:
ecotox_data_modified = pd.DataFrame(ecotox_data[ecotox_data['endpoint'].isin(['LC50', 'ER50', 'ET50', 'NOEL', 'LT50', 'LD50', 'ID50', '(log)LC50 ', 'EC50', 'ED50', 'LR50', 'LOEL', 'NOEC', 'LOEC', 'IC50'])])
print(ecotox_data_modified.shape)
ecotox_data_modified.head()

In [ ]:
ecotox_data_modified = ecotox_data_modified[~(ecotox_data_modified["conc1_mean"].isin(['', 'NR',]))]
print(ecotox_data_modified.shape)
ecotox_data_modified.head()

Unit conversion

Import modified list of units

In [ ]:
units = pd.read_csv('../Data/Unit_conversion_modified_aquaculture.tsv',dtype=str,sep="\t")
print(units.shape)

units = units.replace(np.nan,'',regex=True)
print(units.shape)
units.head()

In [ ]:
# check
print(len(set(ecotox_data_modified['conc1_unit']) - set(units['Unit'])))
set(ecotox_data_modified['conc1_unit']) - set(units['Unit'])

Import aquaculture data

In [ ]:
aquachem_df = pd.read_csv('./external_data/MAIN_chemical_structure_data.tsv',dtype=str,sep='\t')
aquachem_df = aquachem_df.replace(np.nan, '', regex=True)
print(aquachem_df.shape)
aquachem_df.head()

In [ ]:
aquachem_df['PubChemID'] = aquachem_df['PubChemID'].str.replace('CID:', '').str.strip()
aquachem_df['CASRN'] = aquachem_df['CASRN'].str.replace('CAS:', '').str.strip()
print(aquachem_df.shape)
aquachem_df.head()

In [ ]:
aquachems_cas = set(aquachem_df['CASRN']) - {''}
len(aquachems_cas)

In [ ]:
aquachem_df['cas_stripped'] = aquachem_df['CASRN'].str.replace('-', '')
aquachem_df['cas_org'] = aquachem_df['cas_stripped']+'|'+aquachem_df['Organism']
print(aquachem_df.shape)
aquachem_df.head()

In [ ]:
aquachems = set(aquachem_df['cas_stripped']) - {''}
len(aquachems)

In [ ]:
# check
print(len(set(ecotox_data['test_cas'])))
print(len(aquachems.intersection(set(ecotox_data_modified['test_cas']))))

In [ ]:
aquachemspecies_df = pd.read_csv('./external_data/aquaculture_relevant_species.txt', dtype=str, header=None)
print(aquachemspecies_df.shape)

aqua_species = set(list(aquachemspecies_df[0])) - {''}
print(len(aqua_species))
print(len(set([str(x).lower() for x in aqua_species]).intersection(set(ecotox_data_modified['latin_name'].str.lower())))) #check
print(len(aqua_species.intersection(set(ecotox_data_modified['latin_name']))))

In [ ]:
weights_modified = pd.DataFrame(aquachem_df[["CASRN","cas_stripped","Molecular_weight"]])
weights_modified = weights_modified.drop_duplicates()
print(weights_modified.shape)
weights_modified.head()

In [ ]:
#check
weights_modified.drop(columns=['cas_stripped'])['CASRN'].value_counts()

In [ ]:
print(ecotox_data_modified.shape)
ecotox_data_modified = ecotox_data_modified.merge(weights_modified, left_on='test_cas', right_on="cas_stripped", how="left")
ecotox_data_modified = ecotox_data_modified.replace(np.nan, '', regex=True)
print(ecotox_data_modified.shape)
ecotox_data_modified.head()

In [ ]:
ecotox_data_modified.drop(columns=["cas_stripped"], inplace=True)
print(ecotox_data_modified.shape)
ecotox_data_modified.head()

In [ ]:
[x for x in set(ecotox_data_modified['conc1_mean']) if '~' in x]

In [ ]:
# check for zero
sorted([float(x.replace('*','').replace('+','').replace('~-','').strip()) for x in set(ecotox_data_modified['conc1_mean']) if x not in ['NR','']])

If any endpoint has the value zero we will remove it later

In [ ]:
def get_equivalent_ppm(row):
    val = row['conc1_mean'].replace('*','').replace('+','').replace('~-','').strip()
    val_unit = row['conc1_unit']
    weight = row['Molecular_weight']
    
    ppm_converted = units[units['Unit'] == val_unit]['Conversion to ppm'].iloc[0]
    
    if ppm_converted == 'yes':
        conversion_factor = units[units['Unit'] == val_unit]['Converted value equivalent in PPM'].iloc[0].strip()
        if ("MW" in conversion_factor) and (weight!=''):
            conversion_factor_new = conversion_factor.replace('MW',f'*{weight}')
            conversion_factor_new = float(eval(conversion_factor_new))
        elif ("MW" in conversion_factor) and (weight==''):
            conversion_factor_new = '-' # for debugging
        else:
            conversion_factor_new = float(conversion_factor)

        if (val != 'NR') and (conversion_factor_new != '-'):
            ppm_equivalent = float(val)*conversion_factor_new
        elif (val != 'NR') and (conversion_factor_new == '-'):
            ppm_equivalent = '-' # for debugging
        else:
            ppm_equivalent = ''
    else:
        ppm_equivalent = ''
    return (ppm_converted,ppm_equivalent)

In [ ]:
ecotox_data_modified['ppm_converted'] = ecotox_data_modified.apply(lambda row:get_equivalent_ppm(row)[0],axis=1)
ecotox_data_modified['ppm_equivalent'] = ecotox_data_modified.apply(lambda row:get_equivalent_ppm(row)[1],axis=1)

print(ecotox_data_modified.shape)
ecotox_data_modified.head()

In [ ]:
set(ecotox_data_modified[ecotox_data_modified['ppm_converted'] == 'no']['ppm_equivalent'])

In [ ]:
set(ecotox_data_modified[(ecotox_data_modified['ppm_converted'] == 'yes')&(ecotox_data_modified['ppm_equivalent'] == '-')]['CASRN'])

In [ ]:
print(len(set(ecotox_data_modified['CASRN']) - {''}))
print(len(set(ecotox_data_modified['latin_name']) - {''}))

In [ ]:
# keeping only 'yes' for ppm converted and ppm equivalent is not empty
print(ecotox_data_modified.shape)
ecotox_data_modified['With_std_concentrations'] = ecotox_data_modified.apply(lambda row: 'Y' if ((row['ppm_converted'] == 'yes')&(row['ppm_equivalent'] != '')) else 'N', axis=1)
print(ecotox_data_modified.shape)
ecotox_data_modified.head()

In [ ]:
ecotox_data_modified['With_std_concentrations'].value_counts()

## For aquaculture chemical-species combinations

In [ ]:
aquachem_df.columns

In [ ]:
aquachem_df = aquachem_df[['ID', 'Chemical_name', 'PubChemID', 'CASRN', 'cas_stripped']]
print(aquachem_df.shape)
aquachem_df.head()

In [ ]:
len(aqua_species)

For test species

In [ ]:
aqua_sp = pd.DataFrame(ecotox_data_modified[ecotox_data_modified['latin_name'].isin(aqua_species)])
print(aqua_sp.shape)
aqua_sp.head()

In [ ]:
print(set(aqua_sp['ecotox_group_species']))
print(len(set(aqua_sp['latin_name'])))
print(set(aqua_sp['endpoint']))
print(len(set(aqua_sp['endpoint'])))
print(len(set(aqua_sp['test_cas'])))

In [ ]:
# filter a separate list where only data with standard ppm concentrations is present
aqua_sp_std = aqua_sp[aqua_sp['With_std_concentrations']=='Y']
print(aqua_sp_std.shape)
aqua_sp_std.head()

In [ ]:
aqua_sp_std = aqua_sp_std.merge(aquachem_df, left_on='test_cas', right_on='cas_stripped', how='inner')
print(aqua_sp_std.shape)
aqua_sp_std.head()

In [ ]:
aqua_sp_std = aqua_sp_std[aqua_sp_std['endpoint'].isin(['LC50', 'EC50', 'NOEC', 'LOEL', 'NOEL'])]
print(aqua_sp_std.shape)
aqua_sp_std.head()

In [ ]:
list(aqua_sp_std.columns)

In [ ]:
aqua_sp_std = aqua_sp_std[[
'ID',
'Chemical_name',
'PubChemID',
'CASRN_y',
'ecotox_group_chem',
'latin_name',
'ecotox_group_species',
'ncbi_taxid',
'organism_habitat',
'endpoint',
'conc1_mean',
'conc1_unit',
'ppm_equivalent',
'trend',
'trend_description',
'effect',
'effect_description',
'measurement',
'measurement_description',
'ref'
]].drop_duplicates()
print(aqua_sp_std.shape)
aqua_sp_std.head()

In [ ]:
aqua_sp_std['measurement'] = aqua_sp_std['measurement'].replace('/', '', regex=True)

In [ ]:
aqua_sp_std.columns = [
'ID',
'Chemical_name',
'PubChemID',
'CASRN',
'ECOTOX Chemical Group',
'Latin Name',
'ECOTOX Species Group',
'NCBI TaxID',
'Organism Habitat',
'Endpoint',
'Concentration Value',
'Concentration Unit',
'PPM Equivalent',
'Trend',
'Trend Description',
'Effect',
'Effect Description',
'Measurement',
'Measurement Description',
'Reference'
]
print(aqua_sp_std.shape)
aqua_sp_std.head()

In [ ]:
aqua_sp_std['Reference'].value_counts()

In [ ]:
print(set(aqua_sp_std['ECOTOX Species Group']))
print(len(set(aqua_sp_std['Latin Name'])))
print(set(aqua_sp_std['Endpoint']))
print(len(set(aqua_sp_std['Endpoint'])))
print(len(set(aqua_sp_std['CASRN'])))

In [ ]:
aqua_sp_std.to_csv('../Output/FINAL_aquaculture_chemical_toxicity_STDCONC_for_TEST_species.tsv', sep='\t', index=False)

## For food web data

In [ ]:
foodweb = pd.read_csv('../GloBI/Output/food_web_data.tsv',dtype=str, sep='\t')
foodweb = foodweb.replace(np.nan, '', regex=True)
print(foodweb.shape)
foodweb.head()

In [ ]:
len(set(foodweb['source_taxon_name']))

In [ ]:
prey_species = set(foodweb['target_taxon_name'])  - {''}
print(len(prey_species))

print(len(prey_species.intersection(set(ecotox_data_modified['latin_name'])))) 

In [ ]:
aqua_foodweb = pd.DataFrame(ecotox_data_modified[ecotox_data_modified['latin_name'].isin(prey_species)])
print(aqua_foodweb.shape)
aqua_foodweb.head()

In [ ]:
print(set(aqua_foodweb['ecotox_group_species']))
print(len(set(aqua_foodweb['latin_name'])))
print(set(aqua_foodweb['endpoint']))
print(len(set(aqua_foodweb['endpoint'])))

In [ ]:
# filter a separate list where only data with standard ppm concentrations is present
aqua_foodweb_std = aqua_foodweb[aqua_foodweb['With_std_concentrations']=='Y']
print(aqua_foodweb_std.shape)
aqua_foodweb_std.head()

In [ ]:
aqua_foodweb_std = aqua_foodweb_std.merge(aquachem_df, left_on='test_cas', right_on='cas_stripped', how='inner')
print(aqua_foodweb_std.shape)
aqua_foodweb_std.head()

In [ ]:
aqua_foodweb_std = aqua_foodweb_std[aqua_foodweb_std['endpoint'].isin(['LC50', 'EC50', 'NOEC', 'LOEL', 'NOEL'])]
print(aqua_foodweb_std.shape)
aqua_foodweb_std.head()

In [ ]:
aqua_foodweb_std = aqua_foodweb_std[[
'ID',
'Chemical_name',
'PubChemID',
'CASRN_y',
'ecotox_group_chem',
'latin_name',
'ecotox_group_species',
'ncbi_taxid',
'organism_habitat',
'endpoint',
'conc1_mean',
'conc1_unit',
'ppm_equivalent',
'trend',
'trend_description',
'effect',
'effect_description',
'measurement',
'measurement_description',
'ref'
]].drop_duplicates()
print(aqua_foodweb_std.shape)
aqua_foodweb_std.head()

In [ ]:
aqua_foodweb_std['measurement'] = aqua_foodweb_std['measurement'].replace('/', '', regex=True)

In [ ]:
aqua_foodweb_std.columns = [
'ID',
'Chemical_name',
'PubChemID',
'CASRN',
'ECOTOX Chemical Group',
'Latin Name',
'ECOTOX Species Group',
'NCBI TaxID',
'Organism Habitat',
'Endpoint',
'Concentration Value',
'Concentration Unit',
'PPM Equivalent',
'Trend',
'Trend Description',
'Effect',
'Effect Description',
'Measurement',
'Measurement Description',
'Reference'
]
print(aqua_foodweb_std.shape)
aqua_foodweb_std.head()

In [ ]:
print(set(aqua_foodweb_std['ECOTOX Species Group']))
print(len(set(aqua_foodweb_std['Latin Name'])))
print(set(aqua_foodweb_std['Endpoint']))
print(len(set(aqua_foodweb_std['Endpoint'])))
print(len(set(aqua_foodweb_std['CASRN'])))

In [ ]:
aqua_foodweb_std.to_csv('../Output/FINAL_aquaculture_chemical_toxicity_STDCONC_for_FEED_species.tsv', sep='\t', index=False)

## BCF data

In [ ]:
ecotox_data.columns

In [ ]:
print(ecotox_data.shape)
bcf_data = ecotox_data[[
    'result_id', 'test_id', 'reference_number', 'test_cas',
       'sample_size_mean_op', 'sample_size_mean', 'sample_size_unit',
       'obs_duration_mean_op', 'obs_duration_mean', 'obs_duration_unit',
       'endpoint', 'endpoint_comments', 'trend', 'trend_description', 'effect',
       'effect_description', 'effect_comments', 'measurement',
       'measurement_comments', 'measurement_description', 'response_site',
       'response_site_comments', 'effect_pct_mean_op', 'effect_pct_mean',
       'halflife_mean_op', 'halflife_mean', 'halflife_unit', 'bcf1_mean_op',
       'bcf1_mean', 'bcf1_unit', 'bcf2_mean_op', 'bcf2_mean', 'bcf2_unit',
       'bcf3_mean_op', 'bcf3_mean', 'bcf3_unit', 'significance_code',
       'significance_type', 'significance_level_mean_op',
       'significance_level_mean', 'species_number', 'organism_habitat',
       'organism_source', 'organism_source_comments', 'organism_lifestage',
       'organism_lifestage_comments', 'study_duration_mean_op',
       'study_duration_mean', 'study_duration_unit',
       'exposure_duration_mean_op', 'exposure_duration_mean',
       'exposure_duration_unit', 'study_type', 'exposure_type', 'test_type',
       'control_type', 'num_doses_mean_op', 'num_doses_mean', 'media_type',
       'media_type_comments', 'subhabitat', 'subhabitat_description',
       'geographic_code', 'geographic_location', 'water_depth_mean_op',
       'water_depth_mean', 'water_depth_unit', 'latitude', 'longitude',
       'chemical_name', 'dtxsid', 'common_name', 'latin_name',
       'ecotox_group_chem', 'ecotox_group_species', 'ncbi_taxid', 'ref'
]].drop_duplicates()

print(bcf_data.shape)
bcf_data.head()

### For test species

In [ ]:
aqua_bcf = pd.DataFrame(bcf_data[bcf_data['latin_name'].isin(aqua_species)])
print(aqua_bcf.shape)
aqua_bcf.head()

In [ ]:
aqua_bcf['bcf1_mean'].value_counts()

In [ ]:
aqua_bcf = aqua_bcf[~aqua_bcf['bcf1_mean'].isin(['', 'NR'])]
print(aqua_bcf.shape)
aqua_bcf.head()

In [ ]:
aqua_bcf['bcf1_mean'].value_counts()

In [ ]:
print(set(aqua_bcf['ecotox_group_species']))
print(len(set(aqua_bcf['latin_name'])))
print(set(aqua_bcf['endpoint']))
print(len(set(aqua_bcf['endpoint'])))

In [ ]:
aqua_bcf['bcf1_unit'].value_counts()

In [ ]:
def bcf_unit_change(row):
    if row['bcf1_unit'] in ['kg/L', 'g/ml']:
        return 1/float(row['bcf1_mean'])
    else:
        return row['bcf1_mean']


In [ ]:
aqua_bcf['bcf1_mean_modified'] = aqua_bcf.apply(lambda row: bcf_unit_change(row), axis=1)
print(aqua_bcf.shape)
aqua_bcf.head()

In [ ]:
aqua_bcf = aqua_bcf[~aqua_bcf['bcf1_unit'].isin(['--', 'RA'])]
aqua_bcf['bcf1_unit_modified'] = 'L/kg'
print(aqua_bcf.shape)
aqua_bcf.head()

In [ ]:
aqua_bcf.to_csv('../Output/aquaculture_bcf_data.tsv', sep='\t', index=False)

In [ ]:
aqua_bcf = aqua_bcf[aqua_bcf['endpoint'].isin(['BCF'])]
print(aqua_bcf.shape)
aqua_bcf.head()

In [ ]:
aqua_bcf = aqua_bcf.merge(aquachem_df, left_on='test_cas', right_on='cas_stripped', how='inner')
print(aqua_bcf.shape)
aqua_bcf.head()

In [ ]:
aqua_bcf = aqua_bcf[[
'ID',
'Chemical_name',
'PubChemID',
'CASRN',
'ecotox_group_chem',
'latin_name',
'ecotox_group_species',
'ncbi_taxid',
'organism_habitat',
'bcf1_mean_modified',
'bcf1_unit_modified',
'ref'
]].drop_duplicates()
print(aqua_bcf.shape)
aqua_bcf.head()

In [ ]:
aqua_bcf.columns = [
'ID',
'Chemical_name',
'PubChemID',
'CASRN',
'ECOTOX Chemical Group',
'Latin Name',
'ECOTOX Species Group',
'NCBI TaxID',
'Organism Habitat',
'BCF Value',
'BCF Unit',
'Reference'
]
print(aqua_bcf.shape)
aqua_bcf.head()

In [ ]:
print(set(aqua_bcf['ECOTOX Species Group']))
print(len(set(aqua_bcf['Latin Name'])))
print(len(set(aqua_bcf['CASRN'])))

In [ ]:
print(len(set(aqua_sp_std['CASRN'])))
print(len(set(aqua_foodweb_std['CASRN'])))

In [ ]:
aqua_bcf.to_csv('../Output/FINAL_aquaculture_chemical_BCF.tsv', sep='\t', index=False)

### For food web species

In [ ]:
aqua_bcf_prey = pd.DataFrame(bcf_data[bcf_data['latin_name'].isin(prey_species)])
print(aqua_bcf_prey.shape)
aqua_bcf_prey.head()

In [ ]:
aqua_bcf_prey['bcf1_mean'].value_counts()

In [ ]:
aqua_bcf_prey = aqua_bcf_prey[~aqua_bcf_prey['bcf1_mean'].isin(['', 'NR'])]
print(aqua_bcf_prey.shape)
aqua_bcf_prey.head()

In [ ]:
aqua_bcf_prey['bcf1_mean'].value_counts()

In [ ]:
print(set(aqua_bcf_prey['ecotox_group_species']))
print(len(set(aqua_bcf_prey['latin_name'])))
print(set(aqua_bcf_prey['endpoint']))
print(len(set(aqua_bcf_prey['endpoint'])))

In [ ]:
aqua_bcf_prey['bcf1_unit'].value_counts()

In [ ]:
def bcf_unit_change(row):
    if row['bcf1_unit'] in ['kg/L', 'g/ml']:
        return 1/float(row['bcf1_mean'])
    else:
        return row['bcf1_mean']


In [ ]:
aqua_bcf_prey['bcf1_mean_modified'] = aqua_bcf_prey.apply(lambda row: bcf_unit_change(row), axis=1)
print(aqua_bcf_prey.shape)
aqua_bcf_prey.head()

In [ ]:
aqua_bcf_prey = aqua_bcf_prey[~aqua_bcf_prey['bcf1_unit'].isin(['--', 'RA'])]
aqua_bcf_prey['bcf1_unit_modified'] = 'L/kg'
print(aqua_bcf_prey.shape)
aqua_bcf_prey.head()

In [ ]:
aqua_bcf_prey.to_csv('../Output/aquaculture_bcf_prey_species_data.tsv', sep='\t', index=False)

In [ ]:
aqua_bcf_prey = aqua_bcf_prey[aqua_bcf_prey['endpoint'].isin(['BCF'])]
print(aqua_bcf_prey.shape)
aqua_bcf_prey.head()

In [ ]:
aqua_bcf_prey = aqua_bcf_prey.merge(aquachem_df, left_on='test_cas', right_on='cas_stripped', how='inner')
print(aqua_bcf_prey.shape)
aqua_bcf_prey.head()

In [ ]:
aqua_bcf_prey = aqua_bcf_prey[[
'ID',
'Chemical_name',
'PubChemID',
'CASRN',
'ecotox_group_chem',
'latin_name',
'ecotox_group_species',
'ncbi_taxid',
'organism_habitat',
'bcf1_mean_modified',
'bcf1_unit_modified',
'ref'
]].drop_duplicates()
print(aqua_bcf_prey.shape)
aqua_bcf_prey.head()

In [ ]:
aqua_bcf_prey.columns = [
'ID',
'Chemical_name',
'PubChemID',
'CASRN',
'ECOTOX Chemical Group',
'Latin Name',
'ECOTOX Species Group',
'NCBI TaxID',
'Organism Habitat',
'BCF Value',
'BCF Unit',
'Reference'
]
print(aqua_bcf_prey.shape)
aqua_bcf_prey.head()

In [ ]:
print(set(aqua_bcf_prey['ECOTOX Species Group']))
print(len(set(aqua_bcf_prey['Latin Name'])))
print(len(set(aqua_bcf_prey['CASRN'])))

In [ ]:
print(len(set(aqua_sp_std['CASRN'])))
print(len(set(aqua_foodweb_std['CASRN'])))

In [ ]:
aqua_bcf_prey.to_csv('../Output/FINAL_aquaculture_chemical_BCF_feed_species.tsv', sep='\t', index=False)